In [1]:
from pynq.overlays.base import BaseOverlay

from pynq.lib.arduino import Arduino_IO

from pynq.lib.pmod.pmod_pwm import Pmod_PWM
from pynq.lib.pmod.pmod_io import Pmod_IO

import os, time, threading, multiprocessing, logging

base = BaseOverlay("base.bit")

In [2]:
# Helper functions
def _BV(bit):
    return (1 << bit)

def set_motor(speed, pin):
    if speed >= 1:
        pin.write(1)
    elif speed <= 0:
        pin.write(0)
    else:
        for _ in range(100):
            pin.write(1)
            time.sleep(speed * 0.001)
            pin.write(0)
            time.sleep((1-speed) * 0.001)

# Motor -> Arduino header pin mapping
MOTOR_PIN_LATCH  = 12
MOTOR_PIN_CLK    = 4
MOTOR_PIN_ENABLE = 7
MOTOR_PIN_DATA   = 8

# Bit positions in the 74HCT595 shift register output
MOTOR1_A = 2
MOTOR1_B = 3
MOTOR2_A = 1
MOTOR2_B = 4
MOTOR4_A = 0
MOTOR4_B = 6
MOTOR3_A = 5
MOTOR3_B = 7

# Constants that the user passes in to the motor calls
FORWARD  = 1
BACKWARD = 2
BRAKE    = 3
RELEASE  = 4

# Constants that encode the joystick D-pad user input
# This corresponds to map_to_direction() in joystick_controller.ino
DPAD_LEFT           = 1
DPAD_RIGHT          = 2
DPAD_FORWARD        = 3
DPAD_BACKWARD       = 4
DPAD_FORWARD_LEFT   = 5
DPAD_FORWARD_RIGHT  = 6
DPAD_BACKWARD_LEFT  = 7
DPAD_BACKWARD_RIGHT = 8
DPAD_NEUTRAL        = 0

cmd_map = [
    "neutral",
    "left",
    "right",
    "forward",
    "backward",
    "forward_left",
    "forward_right",
    "backward_left",
    "backward_right",
]
    

# PWM rate for DC motors
DC_MOTOR_PWM_RATE = _BV(1) # MOTOR34_8KHZ

latch_state = 0

In [ ]:
%%microblaze base.ARDUINO

// DO NOT RUN THIS AS OF NOW
#include <xio_switch.h>

int pwm_init(void) {
    return 1;
}

In [3]:
class MotorController:
    def __init__(self):
        self.TimerInitalized = False

    def enable(self):
        global latch_state
        self.motor_pin_latch = Arduino_IO(base.ARDUINO, MOTOR_PIN_LATCH, 'out')
        self.motor_pin_enable = Arduino_IO(base.ARDUINO, MOTOR_PIN_ENABLE, 'out')
        self.motor_pin_data = Arduino_IO(base.ARDUINO, MOTOR_PIN_DATA, 'out')
        self.motor_pin_clk = Arduino_IO(base.ARDUINO, MOTOR_PIN_CLK, 'out')

        latch_state = 0
        self.latch_tx() # Reset

        self.motor_pin_enable.write(0)

    def latch_tx(self):
        global latch_state

        self.motor_pin_latch.write(0)
        self.motor_pin_data.write(0)

        for i in range(0, 8):
            self.motor_pin_clk.write(0)

            if (latch_state & _BV(7-i)):
                self.motor_pin_data.write(1)
            else:
                self.motor_pin_data.write(0)

            self.motor_pin_clk.write(1)

        self.motor_pin_latch.write(1)

class DCMotor:
    def __init__(self, motornum, MotorController, freq = DC_MOTOR_PWM_RATE):
        global latch_state

        self.motornum = motornum
        self.pwmfreq = freq
        self.MotorController = MotorController

        MotorController.enable();
        if (motornum == 1):
            self.pwm = Arduino_IO(base.ARDUINO, 11, 'out')
            # set both motor pins to 0
            latch_state &= ~_BV(MOTOR1_A) & ~_BV(MOTOR1_B)
            MotorController.latch_tx()

        elif (motornum == 2):
            self.pwm = Arduino_IO(base.ARDUINO, 3, 'out')
            # set both motor pins to 0
            latch_state &= ~_BV(MOTOR2_A) & ~_BV(MOTOR2_B)
            MotorController.latch_tx()

        elif (motornum == 3):
            self.pwm = Arduino_IO(base.ARDUINO, 6, 'out')
            # set both motor pins to 0
            latch_state &= ~_BV(MOTOR3_A) & ~_BV(MOTOR3_B)
            MotorController.latch_tx()

        elif (motornum == 4):
            self.pwm = Arduino_IO(base.ARDUINO, 5, 'out')
            # set both motor pins to 0
            latch_state &= ~_BV(MOTOR4_A) & ~_BV(MOTOR4_B)
            MotorController.latch_tx()

    def run(self, cmd):
        global latch_state

        if (self.motornum == 1):
            a = MOTOR1_A
            b = MOTOR1_B
        elif (self.motornum == 2):
            a = MOTOR2_A
            b = MOTOR2_B
        elif (self.motornum == 3):
            a = MOTOR3_A
            b = MOTOR3_B
        elif (self.motornum == 4):
            a = MOTOR4_A
            b = MOTOR4_B

        if (cmd == FORWARD):
            latch_state |= _BV(a)
            latch_state &= ~_BV(b)
            self.MotorController.latch_tx()

        elif (cmd == BACKWARD):
            latch_state &= ~_BV(a);
            latch_state |= _BV(b);
            self.MotorController.latch_tx()

        elif (cmd == RELEASE):
            # A and B both low
            latch_state &= ~_BV(a)
            latch_state &= ~_BV(b); 
            self.MotorController.latch_tx()

    def pwm_t(self, speed):
        t = threading.Thread(target=worker_t, args=(forks, i, btn_press_event))

    def set_speed(self, speed):
        set_motor(speed, self.pwm)

In [10]:
class RC_Car:
    def __init__(self, weapons = True, log_level = logging.INFO):
        # Logging
        self.logger = logging.getLogger("PYNQ-Tag")
        self.logger.setLevel(log_level)
        self.logfile_handler = logging.FileHandler("PYNQ-Tag.log", mode="w")

        self.logfile_handler.setFormatter(
            logging.Formatter(
                "%(asctime)s.%(msecs)03d - %(levelname)s - %(message)s",
                datefmt="%Y-%m-%d %H:%M:%S"))
        self.logger.addHandler(self.logfile_handler)

        self.logger.info(f"Beginning program...")
        self.logger.debug(f"Logging initialized")

        # Motors
        self.logger.debug(f"Initializing motors")
        # TODO: Config file?
        # Motor 0 is front-left (M1)
        # Motor 1 is back-left (M2)
        # Motor 2 is back-right (M3)
        # Motor 3 is front-right (M4)
        self.motor_controller = MotorController()
        self.motors = []
        for i in range(1, 5):
            self.motors.append(DCMotor(i, self.motor_controller))
        self.motor_fl = self.motors[0]
        self.motor_fr = self.motors[3]
        self.motor_bl = self.motors[1]
        self.motor_br = self.motors[2]
        self.logger.debug(f"Motors initialized")

        # Weapons
        self.weapons = weapons
        if weapons:
            self.logger.debug(f"Initializing laser and IR receiver")
        else:
            self.logger.debug(f"Skipping laser and IR receiver initialization")
        self.laser = self.Laser(parent_class = self)
        self.ir_receiver = self.IR_Receiver(parent_class = self)
        if weapons:
            self.logger.debug(f"Laser and IR receiver initialized")

    def stop(self):
        self.logger.info(f"Stopping program...")
        self.logfile_handler.close()
        for motor in self.motors:
            motor.run(RELEASE)

    def move(self, cmd):
        global cmd_map
        self.logger.debug(f"Received command: {cmd_map[cmd]}")
        if (cmd == DPAD_FORWARD):
            self.motor_fl.run(FORWARD)
            self.motor_fr.run(FORWARD)
            self.motor_bl.run(FORWARD)
            self.motor_br.run(FORWARD)

        elif (cmd == DPAD_BACKWARD):
            self.motor_fl.run(BACKWARD)
            self.motor_fr.run(BACKWARD)
            self.motor_bl.run(BACKWARD)
            self.motor_br.run(BACKWARD)

        elif (cmd == DPAD_LEFT):
            self.motor_fl.run(BACKWARD)
            self.motor_fr.run(BACKWARD)
            self.motor_bl.run(FORWARD)
            self.motor_br.run(FORWARD)

        elif (cmd == DPAD_RIGHT):
            self.motor_fl.run(FORWARD)
            self.motor_fr.run(FORWARD)
            self.motor_bl.run(BACKWARD)
            self.motor_br.run(BACKWARD)

        elif (cmd == DPAD_FORWARD_LEFT):
            #self.motors[0].set_speed(1)
            #self.motors[2].set_speed(1)
            self.motor_fl.run(FORWARD)
            self.motor_fr.run(FORWARD)
            self.motor_bl.run(FORWARD)
            self.motor_br.run(FORWARD)

        elif (cmd == DPAD_FORWARD_RIGHT):
            #self.motors[1].set_speed(1)
            #self.motors[3].set_speed(1)
            self.motor_fl.run(FORWARD)
            self.motor_fr.run(FORWARD)
            self.motor_bl.run(FORWARD)
            self.motor_br.run(FORWARD)

        elif (cmd == DPAD_BACKWARD_LEFT):
            #self.motors[0].set_speed(1)
            #self.motors[2].set_speed(1)
            self.motor_fl.run(BACKWARD)
            self.motor_fr.run(BACKWARD)
            self.motor_bl.run(BACKWARD)
            self.motor_br.run(BACKWARD)

        elif (cmd == DPAD_BACKWARD_RIGHT):
            #self.motors[1].set_speed(1)
            #self.motors[3].set_speed(1)
            self.motor_fl.run(BACKWARD)
            self.motor_fr.run(BACKWARD)
            self.motor_bl.run(BACKWARD)
            self.motor_br.run(BACKWARD)

        elif (cmd == DPAD_NEUTRAL):
            self.motor_fl.run(RELEASE)
            self.motor_fr.run(RELEASE)
            self.motor_bl.run(RELEASE)
            self.motor_br.run(RELEASE)

    class Laser:
        def __init__(self, parent_class, signal_pin = 3):
            self.logger = parent_class.logger
            self.enable = parent_class.weapons
            if not self.enable:
                self.logger.info(f"Laser not initialized")
                return

            # IR transmitter "laser" is on PMODB and connected to
            # pin 3

            self.pwm = Pmod_PWM(base.PMODB, signal_pin)

            self.pwm_period_usec = 26 # 26 usec is about 38.4 KHz
            self.pwm_duty_cycle = 50

            self.logger.info(f"Initialized Laser class")
            self.logger.debug(f"Laser signal pin: {signal_pin}")
            self.logger.debug(
                f"PWM frequency: {1000.0 / self.pwm_period_usec:.2f} KHz")
            self.logger.debug(f"PWM duty cycle: {self.pwm_duty_cycle}%")

        def shoot(self, pulse_time_msec = 250):
            if not self.enable:
                self.logger.debug(f"Laser could not be shot - not initialized!")
                return

            self.logger.info(f"Laser shot!")
            self.logger.debug(
                f"Laser pulse duration: {pulse_time_msec / 1000} seconds")

            # Shoot a quarter second pulse "laser"
            self.pwm.generate(self.pwm_period_usec, self.pwm_duty_cycle)
            time.sleep(pulse_time_msec / 1000)
            self.pwm.stop()

    class IR_Receiver():
        def __init__(self, parent_class, signal_pin = 3):
            self.enable = parent_class.weapons
            self.logger = parent_class.logger
            if not self.enable:
                self.logger.info(f"IR receiver not initialized")
                return

            # IR receiver is on PMODA and connected to pin 3
            self.io = Pmod_IO(base.PMODA, signal_pin, "in")

            # Start a process which indicates a hit
            self.proc = multiprocessing.Process(
                            target=self.notify_hit_p, args=())
            self.proc.start()
            os.system("taskset -p {} {}".format(0x2, self.proc.pid))

            self.logger.info(f"Initialized IR Receiver class")
            self.logger.debug(f"IR receiver signal pin: {signal_pin}")

        def notify_hit_p(self):
            if not self.enable:
                return

            prev_read = self.io.read()
            while True:
                curr_read = self.io.read()
                if curr_read == 0 and curr_read != prev_read:
                    # TODO: This will send a kill signal
                    self.logger.info("Hit!")
                    print("Hit!")
                prev_pread = curr_read
                time.sleep(0.2)


In [20]:
car = RC_Car(False, logging.DEBUG)
car.laser.shoot()

car.move(DPAD_FORWARD)
time.sleep(0.1)
car.move(DPAD_NEUTRAL)
time.sleep(0.1)
car.move(DPAD_BACKWARD)
time.sleep(0.1)
car.move(DPAD_NEUTRAL)
time.sleep(0.1)
car.move(DPAD_FORWARD)
time.sleep(0.1)
car.move(DPAD_NEUTRAL)
time.sleep(0.1)
car.stop()